# Fixation mRNN Training Smoke Test

Fifty-epoch scratch runs for the `src`-native fixation mRNN workflow. These cells validate data loading, raw firing-rate training, region-PC training, checkpoint replay, feature-order restoration, flow-field analysis, and model diagnostic plots before launching larger indexed experiments.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root")


repo_root = find_repo_root(Path.cwd().resolve())
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

repo_root

In [ ]:
from dal_monte_2022_analysis.ephys.modeling import (
    build_fixation_mrnn_targets,
    compute_fixation_mrnn_currents,
    compute_fixation_mrnn_eigenvalues,
    compute_fixation_mrnn_flow_fields,
    load_fixation_mrnn_config,
    replay_fixation_mrnn_run,
    settings_from_config,
    train_fixation_mrnn_scratch,
)
from dal_monte_2022_analysis.ephys.plotting import (
    FixationMRNNDiagnosticPlotSettings,
    plot_fixation_mrnn_activation_trajectories_3d,
    plot_fixation_mrnn_current_influence,
)

MRNN_CFG = repo_root / "configs" / "ephys_fixation_mrnn.yaml"
cfg = load_fixation_mrnn_config(MRNN_CFG)
settings = settings_from_config(cfg)
settings.dataset_cfg_path = str(repo_root / "configs" / "dataset.yaml")
settings.device = "auto"
settings.epochs = 50
settings.hidden_units = 8
settings.checkpoint_every = 0
settings.seed = 123456
settings

In [ ]:
targets = build_fixation_mrnn_targets(
    settings.dataset_cfg_path,
    input_subdir=settings.input_subdir,
    dataframe_filename=settings.dataframe_filename,
    timeline_filename=settings.timeline_filename,
    canonical_region_order=settings.canonical_region_order,
    normalize_targets=settings.normalize_targets,
    normalization_stabilizer=settings.normalization_stabilizer,
    pca_variance_threshold=settings.pca_variance_threshold,
)

print("conditions", targets.condition_names)
print("input", targets.input_tensor.shape)
print("raw", {k: v.shape for k, v in targets.raw_targets_by_region.items()})
print("pcs", {k: v.shape for k, v in targets.pc_targets_by_region.items()})
print("raw features", {k: len(v) for k, v in targets.raw_feature_order_by_region.items()})
print("pc features", {k: len(v) for k, v in targets.pc_feature_order_by_region.items()})

## Raw Firing-Rate Scratch Run

In [ ]:
raw_settings = settings_from_config(cfg)
raw_settings.dataset_cfg_path = settings.dataset_cfg_path
raw_settings.device = settings.device
raw_settings.epochs = 50
raw_settings.hidden_units = 8
raw_settings.seed = 123456
raw_settings.target_mode = "raw_fr"

raw_result = train_fixation_mrnn_scratch(
    raw_settings,
    scratch_id="smoke_raw_fr",
    overwrite=True,
)
raw_result["run_dir"], raw_result["history"].tail()

In [ ]:
raw_replay = replay_fixation_mrnn_run(raw_result["run_dir"], device="cpu")
print("internal order", raw_replay["checkpoint"]["internal_region_order"])
print("canonical", {k: tuple(v.shape) for k, v in raw_replay["canonical_output_by_region"].items()})

current_df, _ = compute_fixation_mrnn_currents(raw_replay)
eig_df = compute_fixation_mrnn_eigenvalues(raw_replay)
print(current_df.head())
print(eig_df.head())

## Region-PC Scratch Run

In [ ]:
pc_settings = settings_from_config(cfg)
pc_settings.dataset_cfg_path = settings.dataset_cfg_path
pc_settings.device = settings.device
pc_settings.epochs = 50
pc_settings.hidden_units = 8
pc_settings.seed = 223456
pc_settings.target_mode = "region_pcs"

pc_result = train_fixation_mrnn_scratch(
    pc_settings,
    scratch_id="smoke_region_pcs",
    overwrite=True,
)
pc_result["run_dir"], pc_result["history"].tail()

In [ ]:
pc_replay = replay_fixation_mrnn_run(pc_result["run_dir"], device="cpu")
print("internal order", pc_replay["checkpoint"]["internal_region_order"])
print("canonical", {k: tuple(v.shape) for k, v in pc_replay["canonical_output_by_region"].items()})

flow = compute_fixation_mrnn_flow_fields(
    pc_replay,
    region="ofc",
    condition="face_interactive",
    num_points=5,
)
flow["region"], flow["condition"], len(flow["flow_fields"])

## Model Diagnostic Plots

In [ ]:
plot_settings = FixationMRNNDiagnosticPlotSettings()
activation_fig, activation_axes = plot_fixation_mrnn_activation_trajectories_3d(
    pc_replay,
    settings=plot_settings,
)
activation_fig

In [ ]:
current_df, current_vectors = compute_fixation_mrnn_currents(pc_replay)
current_fig, current_axes = plot_fixation_mrnn_current_influence(
    pc_replay,
    current_vectors=current_vectors,
    settings=plot_settings,
)
current_fig